In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [26]:
anos = ['2025-12-31',
 '2024-12-31',
 '2023-12-31',
 '2022-12-31',
 '2021-12-31',
 '2020-12-31',
 '2019-12-31',
 '2018-12-31',
 '2017-12-31',
 '2016-12-31',
 '2015-12-31',
 ]

## LER OS RETORNOS, SCORE e SELIC

In [38]:

# Retornos

dfs = {}
for ano in anos:
    ano_usado = ano.split('-')[0]
    ano_usado_mais_um = int(ano_usado)+1
    nome = f'df_ativos_{ano_usado}.csv'
    # caminho = Path('retornos_ativos_otimizacao') / nome
    dfs[ano_usado] = pd.read_csv('../dados_retornos/otimizacao/ativos/'+nome)
    dfs[ano_usado]['date'] = pd.to_datetime(dfs[ano_usado]['date'])
    dfs[ano_usado] = dfs[ano_usado].set_index('date')
    dfs[ano_usado] = dfs[ano_usado].drop(columns=dfs[ano_usado].columns[(dfs[ano_usado] == 0).all()])
    dfs[ano_usado] = dfs[ano_usado].loc[str(ano_usado_mais_um)]
    print(ano,'-',len(dfs[ano_usado].columns))
    

2025-12-31 - 100
2024-12-31 - 99
2023-12-31 - 99
2022-12-31 - 99
2021-12-31 - 97
2020-12-31 - 84
2019-12-31 - 77
2018-12-31 - 76
2017-12-31 - 73
2016-12-31 - 69
2015-12-31 - 70


In [41]:
dfs['2016']

,ABEV3,ALOS3,ALPA4,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,...,SMTO3,TAEE11,TIMS3,TOTS3,UGPA3,USIM5,VALE3,VIVT3,WEGE3,YDUQ3
date,,,,,,,,,,,,,,,,,,,,,
2017-01-02,-0.005488,-0.013349,-0.016983,-0.027880,-0.013591,-0.000399,-0.013333,-0.019579,-0.010981,-0.006896,...,-0.013846,0.031823,-0.012772,-0.018719,-0.008035,-0.004878,-0.024143,-0.019257,-0.025806,-0.022152
2017-01-03,0.020233,0.001177,0.006098,0.009057,-0.007111,0.034331,0.041155,0.045751,0.040524,0.042352,...,0.006240,0.001870,0.043985,0.051293,0.016201,0.051471,0.044294,0.017812,0.050331,0.055664
2017-01-04,-0.007812,0.005875,0.019192,-0.001496,0.011191,0.028561,0.018879,-0.005208,-0.000334,-0.006333,...,0.001550,-0.008862,0.019827,-0.006048,-0.015942,0.060606,-0.017960,-0.002778,-0.023959,-0.004905
2017-01-05,0.005451,-0.001752,-0.009911,0.003745,-0.003984,0.005628,-0.012159,-0.002443,0.005674,0.011070,...,-0.004128,0.004706,0.012150,0.005274,0.005891,0.010989,0.038132,-0.036490,-0.002584,-0.016020
2017-01-06,-0.008434,0.003511,0.004004,0.029851,-0.001333,-0.001865,-0.000586,0.010846,0.007634,0.006304,...,-0.015026,-0.008431,-0.012005,-0.031073,-0.004392,-0.028261,-0.026612,0.023707,0.008420,-0.007514
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2017-03-27,0.009076,0.005454,-0.023083,-0.015579,-0.054154,-0.012862,-0.001604,0.003313,0.007844,0.008509,...,0.016000,0.004098,-0.005848,-0.015043,0.010913,0.086420,0.013448,-0.011200,0.010431,0.004027
2017-03-28,0.008432,0.004069,0.008439,-0.003768,-0.036131,0.026059,-0.000535,-0.006305,0.011519,0.005625,...,0.009561,0.017234,-0.009804,-0.001455,0.002103,-0.013636,0.014291,-0.013172,0.014724,0.019385
2017-03-29,0.013378,0.006754,0.015900,0.003026,-0.010957,-0.004762,0.036422,0.036254,0.003386,0.020821,...,-0.025070,0.014266,0.001980,-0.025492,0.014270,0.023041,0.010064,0.012814,0.013349,-0.001312


## Hiperparâmetros

In [42]:
vb_cardinalidade_max = 20
# vb_cardinalidade_min = 10
vb_peso_maximo = 0.15
# vb_peso_minimo = 0.02
vb_theta = 0.25

### Fazendo otimização ano a ano e comparando com o proximo ano

In [43]:
anos = ['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']
print(anos)

['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']


In [47]:
y = []

carteiras_anuais = {}

melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])

print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)

for an in anos:
    ano = an.split("-")[0]
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass

    try:
        ano_um = int(ano)
        # df_usado = f'df_ativos_{ano_um}'
        # retorno_usado = pd.DataFrame(eval(df_usado))
        df_usado = dfs[str(ano_um)]
        retorno_usado = df_usado.copy()
        retorno_usado = (retorno_usado - retorno_usado.min())/(retorno_usado.max() - retorno_usado.min())
        print("Retornos atualizados")

        sigma_usado = retorno_usado.cov()
        print("Sigmas Atualizados")
        
        print("-----")
    except Exception as e:
        print("ERRRRRRRRRRRROR")
        print(e)
    print("# ------ CRIAÇÃO DO MODELO")
    print("## Carteira criada para o ano: ",int(ano)+1)

    # print("## UTILIZANDO SCORE DO ANO DE: ",ano)
    print("## UTILIZANDO DADOS DE RETORNO DE: ",str(ano_um))
    # print("## UTILIZANDO EXCESSO DO ANO DE: ",ano_um)
    print("## UTILIZANDO SIGMAS DO ANO DE: ",ano_um)

    model = pyo.ConcreteModel()

    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = retorno_usado.columns.tolist())
    model.ativos = pyo.RangeSet(0, len(retorno_usado.columns.tolist())-1)
    model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    # model.theta = pyo.Param(initialize=vb_theta)
    # model.score = pyo.Param(model.ativos, initialize=lambda model,a: score_usado.iloc[0,a])
    model.cardinalidade_valor_max = pyo.Param(initialize=vb_cardinalidade_max)
    # model.cardinalidade_valor_min = pyo.Param(initialize=vb_cardinalidade_min)
    model.peso_maximo = pyo.Param(initialize=vb_peso_maximo)
    # model.peso_minimo = pyo.Param(initialize=vb_peso_minimo)
    model.x = pyo.Var(model.ativos,bounds=(0,1),domain=pyo.NonNegativeReals)
    model.y = pyo.Var(model.ativos, within=pyo.Binary)
    # model.excesso = pyo.Param( model.ativos , initialize = lambda model,a: excesso_usado.mean().iloc[a])
    model.sigma = pyo.Param(model.ativos, model.ativos, initialize = lambda model,a,b: sigma_usado.iloc[a,b])
    # model.s = pyo.Param(initialize = 2, mutable=True)
    # model.r = pyo.Var(within=pyo.NonNegativeReals)
    
    #-------------------------------------- FUNÇÕES

    #=============================
    # Função Objetivo
    #=============================

    def func_objetivo_1(model):
        return sum(model.x[a]*model.x[b]*model.sigma[a,b] for a in model.ativos for b in model.ativos)
    model.obj1 = pyo.Objective(rule=func_objetivo_1, sense=pyo.minimize)
    #=============================
    # RESTRIÇÕES
    #=============================
    #REstricao 1 x só ativa se y = 1
    def restr_vinculo_x_y(model, a):
        return model.x[a] <= model.y[a]
    model.const_restr_vinculo_x_y = pyo.Constraint(model.ativos, rule=restr_vinculo_x_y)

    #peso maximo por acao
    def rule_peso_maximo(model, a):
        # return model.x[a] <= 1/model.cardinalidade_valor
        return model.x[a] <= model.peso_maximo * model.y[a]  # se y=1, então x <= 0.20
    model.const_peso_maximo = pyo.Constraint(model.ativos, rule=rule_peso_maximo)

    # #peso minimo por acao
    # def rule_peso_minimo(model, a):
    #     return model.x[a] >= model.peso_minimo * model.y[a]  # se y=1, então x >= 0.05
    # model.const_peso_minimo = pyo.Constraint(model.ativos, rule=rule_peso_minimo)

    #Restrição 2 soma peso 1
    def soma_peso_1(model):
        return sum(model.x[a] for a in model.ativos) == 1
    model.const_soma_peso_1 = pyo.Constraint(rule=soma_peso_1)

    # def cardinalidade_min(model):
    #     return sum(
    #         model.y[a] for a in model.ativos
    #         ) >= model.cardinalidade_valor_min
    # model.const_cardinalidade_total_min = pyo.Constraint(rule=cardinalidade_min)

    def cardinalidade_max(model):
        return sum(
            model.y[a] for a in model.ativos
            ) <= model.cardinalidade_valor_max
    model.const_cardinalidade_total_max = pyo.Constraint(rule=cardinalidade_max)


    # NOTEBOOOK
    opt = SolverFactory('cplex', executable='C:\\CPLEX_Studio2211\\cplex\\bin\\x64_win64\\cplex.exe')

    # PC
    # opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
    res = opt.solve(model,tee=False)

    melhor_pesos = {list(model.nome_ativos.data())[a]: pyo.value(model.x[a]) for a in model.ativos}
    carteiras_anuais[int(ano_um)+1] = {
        'pesos':  melhor_pesos,
        # 'sharpe_anual': s_lo*np.sqrt(252),
    }
    print(f"{ano_um} -> {melhor_pesos}")
    print("+-="*30)
    if 'model' in locals():
        del model
        print('DELETADO DPS DO WHILE')
    else:
        print("Nao consta model")
        pass




=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Anos totais:  ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
COMEÇANDO ANO NOVO:  2015
Modelo Antigo deletado 
 Iniciando Novo
Retornos atualizados
Sigmas Atualizados
-----
# ------ CRIAÇÃO DO MODELO
## Carteira criada para o ano:  2016
## UTILIZANDO DADOS DE RETORNO DE:  2015
## UTILIZANDO SIGMAS DO ANO DE:  2015
2015 -> {'ABEV3': 0.0, 'ALOS3': 0.0154630005786661, 'ALPA4': 0.032479984501260835, 'ANIM3': 0.0, 'AXIA3': 0.0, 'AZZA3': 0.0, 'B3SA3': 0.0, 'BBAS3': 0.022357283179459876, 'BBDC3': 0.0, 'BBDC4': 0.0, 'BBSE3': 0.0, 'BEEF3': 0.0, 'BRAP4': 0.0, 'CMIG4': 0.0, 'COGN3': 0.0, 'CPFE3': 0.0, 'CPLE3': 0.0, 'CSAN3': 0.0, 'CSMG3': 0.00880592192158111, 'CSNA3': 0.0, 'CVCB3': 0.0, 'CYRE3': 0.03443060637454701, 'DIRR3': 0.0, 'ECOR3': 0.0, 'EGIE3': 0.0, 'EMBJ3':

In [48]:
carteiras_anuais

{2016: {'pesos': {'ABEV3': 0.0,
   'ALOS3': 0.0154630005786661,
   'ALPA4': 0.032479984501260835,
   'ANIM3': 0.0,
   'AXIA3': 0.0,
   'AZZA3': 0.0,
   'B3SA3': 0.0,
   'BBAS3': 0.022357283179459876,
   'BBDC3': 0.0,
   'BBDC4': 0.0,
   'BBSE3': 0.0,
   'BEEF3': 0.0,
   'BRAP4': 0.0,
   'CMIG4': 0.0,
   'COGN3': 0.0,
   'CPFE3': 0.0,
   'CPLE3': 0.0,
   'CSAN3': 0.0,
   'CSMG3': 0.00880592192158111,
   'CSNA3': 0.0,
   'CVCB3': 0.0,
   'CYRE3': 0.03443060637454701,
   'DIRR3': 0.0,
   'ECOR3': 0.0,
   'EGIE3': 0.0,
   'EMBJ3': 0.12200323628906695,
   'ENEV3': 0.01720567378758514,
   'ENGI11': 0.07445624498056408,
   'EQTL3': 0.0,
   'EZTC3': 0.025839127975657926,
   'FLRY3': 0.0,
   'GGBR4': 0.0,
   'GOAU4': 0.0,
   'HYPE3': 0.0,
   'ISAE4': 0.0,
   'ITSA4': 0.0,
   'ITUB3': 0.0,
   'ITUB4': 0.0,
   'JHSF3': 0.00977490120624958,
   'KLBN11': 0.0,
   'LREN3': 0.0,
   'MBRF3': 0.0,
   'MGLU3': 0.08611276030946947,
   'MOTV3': 0.0,
   'MRVE3': 0.042056906452770215,
   'MULT3': 0.0,
   'NA

In [49]:
linhas = []
for an in anos:
    an = int(an.split('-')[0])+1
    print(an)
    for k, v in carteiras_anuais[an]['pesos'].items():
        if round(v, 4)>0.000 and v <= vb_peso_maximo*1.10:
            linhas.append({'ano': an, 'ativo': k, 'peso': v})

df_portfolios = pd.DataFrame(linhas)

# # visões instantâneas:
# df_portfolios[df_portfolios['ano'] == 2016].sort_values('peso', ascending=False)  # uma carteira
# df_portfolios.pivot(index='ativo', columns='ano', values='peso')                  # matriz ativo × ano
# df_portfolios.groupby('ativo')['ano'].count().sort_values(ascending=False)  


2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026


In [50]:
df_portfolios[df_portfolios['ano'] == 2018].sort_values('peso', ascending=False)['peso']

57    0.150000
55    0.150000
53    0.117386
56    0.086694
49    0.078669
59    0.070571
46    0.060068
40    0.055822
58    0.031015
54    0.030346
51    0.029234
41    0.026722
43    0.023960
45    0.022822
47    0.017282
52    0.014076
48    0.013175
44    0.012965
42    0.006725
50    0.002469
Name: peso, dtype: float64

In [51]:
df_portfolios.to_csv('../carteiras/min_var/carteiras_minvar.csv')
